In [4]:
import torch
import numpy as np
import soundfile as sf
import warnings
import gradio as gr
from transformers import (
    WhisperProcessor, WhisperForConditionalGeneration,
    AutoTokenizer, AutoModelForSeq2SeqLM,
    SpeechT5Processor, SpeechT5ForTextToSpeech, SpeechT5HifiGan
)

warnings.filterwarnings("ignore")

print("🚀 Загрузка моделей... (это займет около минуты)")
device = "cpu"

# 1. STT (Распознавание)
asr_model_id = "openai/whisper-small"
asr_processor = WhisperProcessor.from_pretrained(asr_model_id)
asr_model = WhisperForConditionalGeneration.from_pretrained(asr_model_id, torch_dtype=torch.float32).to(device)

# 2. MT (Перевод)
mt_model_id = "Helsinki-NLP/opus-mt-ru-en"
mt_tokenizer = AutoTokenizer.from_pretrained(mt_model_id)
mt_model = AutoModelForSeq2SeqLM.from_pretrained(mt_model_id, torch_dtype=torch.float32).to(device)

# 3. TTS (Озвучка)
tts_model_id = "microsoft/speecht5_tts"
vocoder_id = "microsoft/speecht5_hifigan"
tts_processor = SpeechT5Processor.from_pretrained(tts_model_id)
tts_model = SpeechT5ForTextToSpeech.from_pretrained(tts_model_id, torch_dtype=torch.float32).to(device)
vocoder = SpeechT5HifiGan.from_pretrained(vocoder_id, torch_dtype=torch.float32).to(device)

def get_speaker_embedding(speaker_index=100):
    """
    Загружает x-vector из датасета cmu-arctic-xvectors.
    speaker_index: 
      - 0-50: преимущественно женские голоса
      - 100-150: преимущественно мужские голоса
    """
    try:
        print(f"⏳ Загрузка голоса диктора #{speaker_index}...")
        from datasets import load_dataset
        
        # КЛЮЧЕВОЕ ИЗМЕНЕНИЕ: trust_remote_code=True
        embeddings_dataset = load_dataset(
            "Matthijs/cmu-arctic-xvectors", 
            split="validation",
            trust_remote_code=True  # ← Это обходит блокировку скриптов
        )
        
        xvector_list = embeddings_dataset["xvector"]
        embedding = torch.tensor(xvector_list[speaker_index], dtype=torch.float32).unsqueeze(0)
        
        print(f"✅ Голос #{speaker_index} успешно загружен!")
        return embedding
    except Exception as e:
        print(f"⚠️ Не удалось загрузить датасет: {e}")
        print("   Используем нейтральный голос (fallback)")
        return torch.zeros((1, 512), dtype=torch.float32)

# Загружаем голос диктора (100 = мужской, 0 = женский)
speaker_embeddings = get_speaker_embedding(speaker_index=100)
speaker_embeddings = speaker_embeddings.to(device)

print("✅ Модели готовы! Запускаю веб-интерфейс...\n")

# ==========================================
# ГЛАВНАЯ ФУНКЦИЯ ОБРАБОТКИ
# ==========================================
def process_speech(audio_input):
    if audio_input is None:
        return "Аудио не записано", "Нет перевода", None
    
    sample_rate, audio_data = audio_input
    
    # === ДИАГНОСТИКА: смотрим, что реально прислал Gradio ===
    print(f"📊 Получено аудио: shape={audio_data.shape}, dtype={audio_data.dtype}, sr={sample_rate}")
    print(f"   Диапазон значений: [{audio_data.min():.4f}, {audio_data.max():.4f}]")
    
    # === ШАГ 1: Конвертация в float32 (КРИТИЧНО!) ===
    # Если Gradio прислал int16 (значения от -32768 до 32767), нормализуем в [-1, 1]
    if audio_data.dtype in ['int16', 'int32']:
        audio_data = audio_data.astype(np.float32) / 32768.0
        print(f"   ✅ Конвертировано из int16 в float32")
    elif audio_data.dtype != np.float32:
        audio_data = audio_data.astype(np.float32)
    
    # === ШАГ 2: Превращаем стерео в моно (если нужно) ===
    if audio_data.ndim > 1:
        audio_data = audio_data.mean(axis=1)  # Усредняем каналы
        print(f"   ✅ Стерео → Моно")
    
    # === ШАГ 3: Ресемплинг до 16000 Гц (требование Whisper) ===
    if sample_rate != 16000:
        import librosa
        audio_data = librosa.resample(audio_data, orig_sr=sample_rate, target_sr=16000)
        sample_rate = 16000
        print(f"   ✅ Ресемплинг до 16000 Гц")
    
    # === ШАГ 4: Финальная проверка громкости ===
    max_amplitude = np.abs(audio_data).max()
    print(f"   📈 Максимальная амплитуда после обработки: {max_amplitude:.4f}")
    
    if max_amplitude < 0.001:
        return "⚠️ Слишком тихая запись", "Too quiet", None
    
    print("🎤 Обработка аудио нейросетью...")
    
    # 1. Распознавание (STT)
    input_features = asr_processor(
        audio_data, 
        sampling_rate=16000, 
        return_tensors="pt"
    ).input_features.to(device)
    
    with torch.no_grad():
        predicted_ids = asr_model.generate(
            input_features, 
            language="ru", 
            task="transcribe"
        )
    russian_text = asr_processor.batch_decode(predicted_ids, skip_special_tokens=True)[0].strip()
    
    if not russian_text or russian_text == "":
        return "Не удалось распознать речь", "Speech not recognized", None

    # 2. Перевод (MT)
    inputs_mt = mt_tokenizer(russian_text, return_tensors="pt").to(device)
    with torch.no_grad():
        translation_ids = mt_model.generate(
            **inputs_mt, 
            max_length=100, 
            num_beams=4, 
            length_penalty=1.0
        )
    english_text = mt_tokenizer.decode(translation_ids[0], skip_special_tokens=True).strip()

    # 3. Озвучка (TTS)
    tts_inputs = tts_processor(text=english_text, return_tensors="pt").to(device)
    with torch.no_grad():
        speech = tts_model.generate_speech(
            tts_inputs["input_ids"], 
            speaker_embeddings, 
            vocoder=vocoder
        )
    
    output_path = "output_translation.wav"
    sf.write(output_path, speech.cpu().numpy(), samplerate=16000)

    return russian_text, english_text, output_path

# ==========================================
# СОЗДАНИЕ ИНТЕРФЕЙСА
# ==========================================
with gr.Blocks(title="AI Real-time Переводчик", theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🎙️ AI Переводчик речи в реальном времени")
    gr.Markdown("Нажми 'Записать', скажи фразу на русском, и нейросеть переведет и озвучит её на английском.")
    
    with gr.Row():
        with gr.Column():
            gr.Markdown("### 1. Твой голос (RU)")
            audio_input = gr.Audio(sources=["microphone"], type="numpy", label="Запиши речь")
            btn_process = gr.Button("🚀 Перевести и озвучить", variant="primary")
            
        with gr.Column():
            gr.Markdown("### 2. Результат (EN)")
            text_ru = gr.Textbox(label="Распознанный текст", lines=2)
            text_en = gr.Textbox(label="Перевод", lines=2)
            audio_output = gr.Audio(label="Озвученный перевод", type="filepath")

    btn_process.click(
        fn=process_speech,
        inputs=[audio_input],
        outputs=[text_ru, text_en, audio_output]
    )

# Запуск приложения
demo.launch(inbrowser=True)

🚀 Загрузка моделей... (это займет около минуты)


Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/396 [00:00<?, ?it/s]

[transformers] SpeechT5ForTextToSpeech LOAD REPORT from: microsoft/speecht5_tts
Key                                         | Status     |  | 
--------------------------------------------+------------+--+-
speecht5.decoder.prenet.encode_positions.pe | UNEXPECTED |  | 
speecht5.encoder.prenet.encode_positions.pe | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
[transformers] You are using a model of type `hifigan` to instantiate a model of type `speecht5_hifigan`. This may be expected if you are loading a checkpoint that shares a subset of the architecture (e.g., loading a `sam2_video` checkpoint into `Sam2Model`), but is otherwise not supported and can yield errors. Please verify that the checkpoint is compatible with the model you are instantiating.


Loading weights:   0%|          | 0/158 [00:00<?, ?it/s]

⏳ Загрузка голоса диктора #100...


`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'Matthijs/cmu-arctic-xvectors' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


⚠️ Не удалось загрузить датасет: Dataset scripts are no longer supported, but found cmu-arctic-xvectors.py
   Используем нейтральный голос (fallback)
✅ Модели готовы! Запускаю веб-интерфейс...

* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.


📊 Получено аудио: shape=(738233,), dtype=int16, sr=44100
   Диапазон значений: [-22537.0000, 26180.0000]
   ✅ Конвертировано из int16 в float32
   ✅ Ресемплинг до 16000 Гц
   📈 Максимальная амплитуда после обработки: 0.7985
🎤 Обработка аудио нейросетью...
